In [ ]:
import warnings
import sys
import os
sys.path.append(os.path.abspath("path"))
warnings.filterwarnings('ignore')
from pm4py.analysis import check_is_workflow_net, check_soundness
from fedap_utils import *
from pm4py.algo.discovery.alpha.variants.plus import apply_aggr

### Preprocessing (EL Filtering)

In [ ]:
pa_el, pa_df = load_convert_el(el_path=r"./data/proact.csv", case_id='ID', activity_key='Event', timestamp_key='Date')
print(pa_df['Event'].unique())
list_events = ['Onset', 'Start_trial', 'M_0000', 'Censored','M_0100', 'Dead', 'M_1000',
 'M_1010', 'M_1001', 'M_0001', 'M_0110', 'M_1100', 'M_0010',
 'M_0101', 'M_0011']
pa_df_filt = pa_df[pa_df['Event'].isin(list_events)]

threshold=100
event_count = pa_df_filt['Event'].value_counts()
pa_df_filt = pa_df_filt[pa_df_filt['Event'].isin(event_count[event_count >= threshold].index)]
pa_df_filt = pa_df_filt[pa_df_filt['ID'] != 544698]
pa_df_filt.to_csv('./data/pa_df_filtered.csv', index=False)

# RWD Testing

### ELs Splitting

In [ ]:
import numpy as np

id_list = pa_df_filt['ID'].unique()

random_seed = 42 # For reproducibility
np.random.seed(random_seed)
np.random.shuffle(id_list)

num_of_splits = 10
id_groups = np.array_split(id_list, num_of_splits)

df_path_list = []
for i, id_split in enumerate(id_groups):
	splitted = pa_df_filt[pa_df_filt['ID'].isin(id_split)]
	path = f"./data/proact_split{i}.csv"
	df_path_list.append(path)
	splitted.to_csv(path_or_buf=path, index=False)

event_log_tot, df_tot = load_convert_el(el_path=r"./data/pa_df_filtered.csv", case_id='ID', activity_key='Event', timestamp_key='Date')
el_splits = []
for path_split in df_path_list:
	el_split, _ = load_convert_el(el_path=path_split, case_id='ID', activity_key='Event', timestamp_key='Date')
	el_splits.append(el_split)

### AA+ on the entire event log

In [ ]:
print('\nEntire Event Log (Alpha+):')
net_tot, initial_marking_tot, final_marking_tot, footprint_matrix_tot, loop_one_list_tot, oneL_inputs_tot, oneL_outputs_tot = apply_monocentric(event_log_tot)

Soundness Check Using Inductive Miner

In [ ]:
net_ind, initial_marking_ind, final_marking_ind = pm4py.discover_petri_net_inductive(event_log_tot)
sound_check_ind, _ = check_soundness(net_ind, initial_marking_ind, final_marking_ind, print_diagnostics=True)
print(f'Soundness Check using Inductive Miner {sound_check_ind}')
pm4py.view_petri_net(net_ind, initial_marking_ind, final_marking_ind)


## Processing

### Distributed Local processing
FMs are computed independently over different event log splits

In [ ]:
FTs, oneL_inputs_dic, oneL_outputs_dic, loop_one_tot_list = simulate_nodes_computation(el_splits=el_splits)

### Master Node

Collected footprint matrices are aggregated and used along with 1-length loop information to run the AA+ algorithm in the master node

In [ ]:
sum_FT, events = pm4py.algo.discovery.alpha.variants.plus.aggregate_fms(fts=FTs)

print('Aggregated Footprint Matrix:\n')
display(sum_FT)

print('Total Footprint Matrix:\n')
display(footprint_matrix_tot)

print(f'Equality Check between Total Footprint and Aggregated Footprint Matrix: {footprint_matrix_tot.equals(sum_FT)}\n')

assert footprint_matrix_tot.equals(sum_FT)

#### AA+

In [ ]:
net_final, initial_marking_final, final_marking_final = apply_aggr(fm=sum_FT,
                                                                   events=events,
                                                                   oneL_inputs=oneL_inputs_dic,
                                                                   oneL_outputs=oneL_outputs_dic,
                                                                   loop_one_list=loop_one_list_tot
                                                                   )

pm4py.view_petri_net(net_final, initial_marking_final, final_marking_final)
pm4py.view_petri_net(net_tot, initial_marking_tot, final_marking_tot)

# In Vitro Testing
Same procedure is repeated using in vitro EL. EL.csv is splitted in A,B and C subsplits each containing a subset of loops.

In [ ]:
event_log_tot_vitro, _ = load_convert_el(el_path=r"./data/EL.csv", case_id='Case_ID', activity_key='Activity', timestamp_key='Timestamp')

el_splits_vitro = []
event_log_A, _ = load_convert_el(el_path=r"./data/EL_A.csv", case_id='Case_ID', activity_key='Activity', timestamp_key='Timestamp')
event_log_B, _ = load_convert_el(el_path=r"./data/EL_B.csv", case_id='Case_ID', activity_key='Activity', timestamp_key='Timestamp')
event_log_C, _ = load_convert_el(el_path=r"./data/EL_C.csv", case_id='Case_ID', activity_key='Activity', timestamp_key='Timestamp')
el_splits_vitro.append(event_log_A)
el_splits_vitro.append(event_log_B)
el_splits_vitro.append(event_log_C)

In [ ]:
print('\nEntire Event Log (Alpha+):')
net_tot_vitro, initial_marking_tot_vitro, final_marking_tot_vitro, footprint_matrix_tot_vitro, loop_one_list_tot_vitro, oneL_inputs_tot_vitro, oneL_outputs_tot_vitro= apply_monocentric(event_log_tot_vitro)

In [ ]:
FTs_vitro, oneL_inputs_dic_vitro, oneL_outputs_dic_vitro, loop_one_list_vitro = simulate_nodes_computation(el_splits=el_splits_vitro)
sum_FT_vitro, events_vitro = pm4py.algo.discovery.alpha.variants.plus.aggregate_fms(fts=FTs_vitro)
net_vitro, initial_marking_vitro, final_marking_vitro = apply_aggr(fm=sum_FT_vitro,
                                                                   events=events_vitro,
                                                                   oneL_inputs=oneL_inputs_dic_vitro,
                                                                   oneL_outputs=oneL_outputs_dic_vitro,
                                                                   loop_one_list=loop_one_list_vitro
                                                                   )
assert sum_FT_vitro.equals(footprint_matrix_tot_vitro)
pm4py.view_petri_net(net_vitro, initial_marking_vitro, final_marking_vitro)